In [2]:
############################################################
# Estimation of Grid States (Julia Version)
# Based on "Extended State Estimation - Minimal Model"
# Original Author (MATLAB): Patrick Panciatici
############################################################

using CSV
using DataFrames              
using LinearAlgebra   
using Distributions    
using Random

############################################################
# Load the results of the time-domain simulation
############################################################

eventType = "lineTrip"  # or "loadStep2", etc.
path = joinpath("Simulation_Minimalmodel", "Res_Sim_$(eventType).csv")

sim_data = CSV.read(path, DataFrame)

t = sim_data.time
theta = hcat(sim_data.theta1, sim_data.theta2)
omega = hcat(sim_data.omega1, sim_data.omega2)
Tm = hcat(sim_data.Tm1, sim_data.Tm2)
N  = sim_data.N
PL = hcat(sim_data.PL1, sim_data.PL2)
PL0   = [unique(sim_data.PL0_1)[1], unique(sim_data.PL0_2)[1]]
P0    = [unique(sim_data.P0_1)[1], unique(sim_data.P0_2)[1]]
Pmax  = [unique(sim_data.Pmax_1)[1], unique(sim_data.Pmax_2)[1]]
KL_nom = unique(sim_data.KL_nom)[1]
KL_one = unique(sim_data.KL_one)[1] 
t_line_trip = unique(sim_data.t_line_trip)[1] 

############################################################
# Sampling configuration
############################################################

Tg = 10.0
nt = 7  # 0,10,20,30,40,50,60

time_s  = zeros(nt)
theta_s = zeros(nt, 2)
omega_s = zeros(nt, 2)
PL_s    = zeros(nt, 2)
Tm_s    = zeros(nt, 2)

time_s[1] = t[1]
time_s[end] = t[end]

theta_s[1, :] .= theta[1, :]
theta_s[end, :] .= theta[end, :]

omega_s[1, :] .= omega[1, :]
omega_s[end, :] .= omega[end, :]

PL_s[1, :] .= PL[1, :]
PL_s[end, :] .= PL[end, :]

Tm_s[1, :] .= Tm[1, :]
Tm_s[end, :] .= Tm[end, :]

for k in 1:nt-2
    Ib = findlast(t .< k*Tg)
    Ia = findfirst(t .> k*Tg)
    time_s[k+1] = k*Tg
    Ctg = Tg / (t[Ia] - t[Ib])
    theta_s[k+1, :] .= theta[Ib, :] .+ (theta[Ia, :] .- theta[Ib, :]) .* Ctg
    omega_s[k+1, :] .= omega[Ib, :] .+ (omega[Ia, :] .- omega[Ib, :]) .* Ctg
    PL_s[k+1, :]    .= PL[Ib, :] .+ (PL[Ia, :] .- PL[Ib, :]) .* Ctg
    Tm_s[k+1, :]    .= Tm[Ib, :] .+ (Tm[Ia, :] .- Tm[Ib, :]) .* Ctg
end

############################################################
# Create the measurement set
############################################################

function Create_measure(time, theta, omega, PL, Tm, PL0, Pmax, KL_nom, KL_one, t_line_trip)
    nt = length(time)
    ρ = 1/100

    std_PL = ρ .* PL0'
    PL_m = PL .+ std_PL .* randn(size(PL))

    PG = Tm .* omega
    std_PG = ρ .* Pmax'
    PG_m = PG .+ std_PG .* randn(size(PG))

    F12 = zeros(nt)
    for k in 1:nt
        KL = (time[k] ≥ t_line_trip) ? KL_one : KL_nom
        F12[k] = KL * (theta[k, 1] - theta[k, 2])
    end

    std_F12 = 10.0
    F12_m = F12 .+ randn(nt) .* std_F12
    std_F21 = 10.0
    F21_m = -F12 .+ randn(nt) .* std_F21

    return (
        PL_m = PL_m, std_PL = std_PL,
        PG_m = PG_m, std_PG = std_PG,
        F12_m = F12_m, std_F12 = std_F12,
        F21_m = F21_m, std_F21 = std_F21
    )
end

MS = Create_measure(time_s, theta_s, omega_s, PL_s, Tm_s, PL0, Pmax, KL_nom, KL_one, t_line_trip)

############################################################
# Linear estimator (MLS_LEQ)
############################################################

function MLS_LEQ(MS)
    # Covariance matrix of measurement noises
    S = zeros(6,6)
    S[1,1] = (MS.std_F12)^2
    S[2,2] = (MS.std_F21)^2
    S[3,3] = (MS.std_PG[1])^2
    S[4,4] = (MS.std_PG[2])^2
    S[5,5] = (MS.std_PL[1])^2
    S[6,6] = (MS.std_PL[2])^2

    # Linear constraints A·X = 0
    A = zeros(3,6)
    A[1,1] = 1; A[1,2] = 1
    A[2,1] = -1; A[2,3] = 1; A[2,5] = -1
    A[3,2] = -1; A[3,4] = 1; A[3,6] = -1

    # Estimate gain matrix G
    G = I - (S*A') * inv(A*S*A') * A

    # Covariance of the estimates
    Cov = G * S

    nt = length(MS.F12_m)
    Estimates = zeros(nt, 6)

    # Loop over all time steps
    for k in 1:nt
        Y = [
            MS.F12_m[k];
            MS.F21_m[k];
            MS.PG_m[k,1];
            MS.PG_m[k,2];
            MS.PL_m[k,1];
            MS.PL_m[k,2];
        ]
        Estimates[k, :] .= G * Y
    end

    return Estimates, Cov
end

GridEstimates, CovEst = MLS_LEQ(MS)

############################################################
# Display results
############################################################

println("Grid state estimation complete.")
println("Covariance matrix of estimates:")
println(CovEst)


Grid state estimation complete.
Covariance matrix of estimates:
[23.52410995944121 -23.52410995944117 20.27940513744931 -11.762054979720595 -3.2447048219918893 11.762054979720595; -23.52410995944116 23.52410995944122 -20.279405137449302 11.762054979720595 3.2447048219918884 -11.762054979720595; 20.27940513744931 -20.279405137449302 31.2753492564218 -10.139702568724651 10.995944118972512 10.139702568724651; -11.762054979720595 11.762054979720595 -10.139702568724651 23.881027489860298 1.6223524109959442 12.118972510139704; -3.2447048219918893 3.244704821991888 10.995944118972512 1.6223524109959442 14.240648940964398 -1.6223524109959442; 11.762054979720595 -11.762054979720595 10.139702568724651 12.118972510139704 -1.6223524109959442 23.881027489860298]
